# Snippet from Cookbook.md


In [ ]:
import pytest
from compitum import Router, Config
@pytest.fixture
def router():
    config = Config.from_yaml('configs/test.yaml')
    return Router(config)
def test_smoke_basic_route(router):
    result = router.route("Test prompt")
    cert = result.certificate
    assert cert['feasible'], "Route infeasible"
    assert cert['drift_signal'] < 0, "Energy not descending"
    assert cert['entropy'] > 1.0, "Entropy too low"
def test_smoke_constraint_enforcement(router):
    result = router.route("Test prompt")
    cert = result.certificate
    for name, data in cert['constraints'].items():
        assert data['slack'] >= -0.01, f"Violation: {name}"
def test_smoke_convergence_trend(router):
    drifts = []
    for _ in range(5):
        result = router.route(f"Prompt {_}")
        drifts.append(result.certificate['drift_signal'])
    avg_drift = sum(drifts) / len(drifts)
    assert avg_drift < -0.005, "Not converging"
@pytest.mark.parametrize("prompt", [
    "Simple question",
    "Complex multi-part query with context",
    "Short",
])
def test_smoke_prompt_variety(router, prompt):
    result = router.route(prompt)
    assert result.certificate['feasible']
